In [ ]:
# ── Cell 1: Imports & Parameters ─────────────────────────────────────────────
# Notebook 14: Feature Extraction — DDIM Synthetic Datasets
# Extract MobileNetV2 features from DDIM synthetic train and val splits
# for FashionMNIST, PathMNIST, and EuroSAT
# Features saved to data/extracted_features/seed_42/ddim/

import os
import numpy as np
import tensorflow as tf

SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)

TARGET_SIZE = 224
BATCH_SIZE  = 64

PROJECT_DIR  = '/content/drive/MyDrive/thesis_project'
SYNTH_DIR    = os.path.join(PROJECT_DIR, 'data', 'synthetic', 'seed_42', 'ddim')
FEATURES_DIR = os.path.join(PROJECT_DIR, 'data', 'extracted_features', 'seed_42', 'ddim')
os.makedirs(FEATURES_DIR, exist_ok=True)

DATASETS = {
    'fashion_mnist': {'file': 'fashion_mnist/fashion_mnist_synthetic.npz', 'channels': 1},
    'path_mnist':    {'file': 'path_mnist/path_mnist_synthetic.npz',       'channels': 3},
    'eurosat':       {'file': 'eurosat/eurosat_synthetic.npz',              'channels': 3},
}

In [ ]:
# ── Cell 2: Load MobileNetV2 ──────────────────────────────────────────────────

base_model = tf.keras.applications.MobileNetV2(
    input_shape=(TARGET_SIZE, TARGET_SIZE, 3),
    include_top=False,
    pooling='avg',
    weights='imagenet'
)
base_model.trainable = False
print('MobileNetV2 loaded — output shape: (N, 1280)')

In [ ]:
# ── Cell 3: Feature Extraction Function ──────────────────────────────────────

def extract_features(X, model, batch_size=64):
    features = []
    n = len(X)
    for start in range(0, n, batch_size):
        batch = X[start:start+batch_size].astype('float32') / 255.0
        batch = tf.image.resize(batch, [TARGET_SIZE, TARGET_SIZE]).numpy()
        if batch.shape[-1] == 1:
            batch = np.repeat(batch, 3, axis=-1)
        batch = tf.keras.applications.mobilenet_v2.preprocess_input(batch * 255.0)
        features.append(model.predict(batch, verbose=0))
        if (start // batch_size) % 10 == 0:
            print(f'  {start}/{n}')
    return np.concatenate(features, axis=0)

In [ ]:
# ── Cell 4: Extract & Save Features for All DDIM Datasets ────────────────────
# Only train and val splits — no synthetic test set exists

for name, cfg in DATASETS.items():
    print(f'\n=== {name} ===')
    data = np.load(os.path.join(SYNTH_DIR, cfg['file']))

    for split in ['X_train', 'X_val']:
        X = data[split]
        print(f'  Extracting {split}: {X.shape}')
        F = extract_features(X, base_model, batch_size=BATCH_SIZE)
        out_path = os.path.join(FEATURES_DIR, f'{name}_{split.lower()}_features.npy')
        np.save(out_path, F)
        print(f'  Saved → {out_path}  shape: {F.shape}')

print('\nAll DDIM synthetic features extracted and saved.')